# Nemotron Reasoning — Baseline SFT Notebook w/ Unsloth

Unsloth is an optimization library that makes finetuning faster and uses lower VRAM. Using Unsloth, we can actually train on the whole dataset within a reasonable time (~1hr/epoch).

Fun fact: Unsloth has an actual guide for Nemotron 3 models - https://unsloth.ai/docs/models/nemotron-3

For training with this notebook, you will need a bunch of libraries other than Unsloth, the most important of which is mamba-ssm

## 1. Setup

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes 
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


In [ ]:
import os

# Reduces CUDA fragmentation OOMs in long runs (PyTorch 2.0+).
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import re
import math
from pathlib import Path
import zipfile
import pandas as pd
from datasets import Dataset
import kagglehub
import torch
from torch import nn
import bitsandbytes as bnb
from unsloth import FastLanguageModel
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"

ADAPTER_DIR = "nemotron-lora-adapter"
SUBMISSION_ZIP = "submission.zip"

FRESH_START = True

if FRESH_START:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
else:
    MODEL_PATH = "/kaggle/input/datasets/mayukh18/nemotron-lora-submission/submission/nemotron-lora-adapter"

LOAD_IN_4BIT = False

# LoRA config
LORA_RANK    = 32
LORA_ALPHA   = 16
LORA_DROPOUT = 0

# ── SFT Configuration ──
MAX_SEQ_LEN = 2048
NUM_EPOCHS  = 1
BATCH_SIZE  = 2
GRAD_ACCUM  = 12
LR          = 2e-4

# ── GRPO Configuration ──
GRPO_SUBSAMPLE_SIZE = 500       # prompts for GRPO (each generates multiple completions)
GRPO_NUM_GENERATIONS = 4        # completions per prompt
GRPO_MAX_COMPLETION = 2048      # max tokens per completion
GRPO_LR = 5e-6                  # lower than SFT
GRPO_EPOCHS = 1
GRPO_TEMPERATURE = 0.7          # diversity for exploration (eval is greedy)

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.0

print("Config ready.")

## 2. Load & Inspect Data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train: {len(train_df):,} rows — columns: {list(train_df.columns)}")
print(f"Test:  {len(test_df):,} rows  — columns: {list(test_df.columns)}")
train_df.head()

In [ ]:
# Quick look at prompt length distribution
train_df["prompt_len"] = train_df["prompt"].str.len()
print(train_df["prompt_len"].describe())

# Sample one puzzle
sample = train_df.sample(1).iloc[0]
print("\n--- Sample Prompt ---")
print(sample["prompt"])
print("\n--- Answer ---")
print(sample["answer"])

## 3. Deterministic Solvers + CoT Data Formatting

Instead of using an external LLM to generate CoT, we use **hand-coded deterministic solvers** for each puzzle type. Each solver:
1. Parses the puzzle structure from the prompt
2. Computes the answer programmatically
3. Generates a step-by-step reasoning trace (CoT)

For puzzles the solvers can't fully crack, a scaffold template is used with the gold answer.

After formatting SFT data, we also build a GRPO dataset (prompt-only, no gold answer in the text) and define **3 reward functions** for the RL phase.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from itertools import combinations

BOXED_INSTRUCTION = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

# ═══════════════════════════════════════════════════════════════════════════════
# DETERMINISTIC SOLVERS — Generate exact answers + detailed reasoning traces
# ═══════════════════════════════════════════════════════════════════════════════

def _round2_candidates(val):
    """Multiple rounding strategies to match answer format."""
    candidates = set()
    candidates.add(f"{round(val, 2):.2f}")
    d = Decimal(str(val)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    candidates.add(str(d))
    candidates.add(f"{math.floor(val * 100) / 100:.2f}")
    candidates.add(f"{math.ceil(val * 100) / 100:.2f}")
    for c in list(candidates):
        if c.endswith('0') and '.' in c:
            candidates.add(c.rstrip('0').rstrip('.'))
    return candidates

def solve_gravity(prompt, answer):
    pairs = re.findall(r't\s*=\s*([\d.]+)\s*s.*?distance\s*=\s*([\d.]+)\s*m', prompt)
    query_t_m = re.search(r'falling distance for t\s*=\s*([\d.]+)\s*s', prompt)
    if not pairs or not query_t_m:
        return None, None
    gs = [2 * float(d) / (float(t)**2) for t, d in pairs]
    g_avg = sum(gs) / len(gs)
    t_q = float(query_t_m.group(1))
    best_g = g_avg
    val = 0.5 * g_avg * t_q**2
    candidates = _round2_candidates(val)
    if answer not in candidates:
        for gi in gs:
            vi = 0.5 * gi * t_q**2
            if answer in _round2_candidates(vi):
                best_g = gi
                val = vi
                break
    predicted = f"{val:.2f}"
    if answer in _round2_candidates(val):
        predicted = answer
    g_lines = "\n".join(f"  Example {i+1}: g = 2 * {d} / {t}^2 = {2*float(d)/float(t)**2:.4f}"
                        for i, (t, d) in enumerate(pairs))
    cot = (
        f"I need to find the hidden gravitational constant g from the examples using d = 0.5 * g * t^2, so g = 2d / t^2.\n\n"
        f"Computing g from each example:\n{g_lines}\n\n"
        f"Average g = {best_g:.4f}\n\n"
        f"For t = {t_q}s:\n"
        f"d = 0.5 * {best_g:.4f} * {t_q}^2 = 0.5 * {best_g:.4f} * {t_q**2:.4f} = {predicted}\n\n"
        f"\\boxed{{{predicted}}}"
    )
    return predicted, cot

def solve_unit_conversion(prompt, answer):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    query_m = re.search(r'convert the following measurement:\s*([\d.]+)\s*m', prompt)
    if not pairs or not query_m:
        return None, None
    ratios = [float(out) / float(inp) for inp, out in pairs]
    ratio = sum(ratios) / len(ratios)
    q = float(query_m.group(1))
    val = ratio * q
    predicted = f"{val:.2f}"
    candidates = _round2_candidates(val)
    if answer not in candidates:
        for r in ratios:
            if answer in _round2_candidates(r * q):
                ratio = r
                val = r * q
                break
    if answer in _round2_candidates(val):
        predicted = answer
    ratio_lines = "\n".join(f"  Example {i+1}: {out} / {inp} = {float(out)/float(inp):.4f}"
                            for i, (inp, out) in enumerate(pairs))
    cot = (
        f"I need to find the secret conversion factor from the examples.\n\n"
        f"Computing ratio (output / input) for each example:\n{ratio_lines}\n\n"
        f"Conversion factor = {ratio:.4f}\n\n"
        f"For input {q} m:\n"
        f"Result = {ratio:.4f} * {q} = {predicted}\n\n"
        f"\\boxed{{{predicted}}}"
    )
    return predicted, cot

def _int_to_roman(num):
    vals = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
    syms = ['M', 'CM', 'D', 'CD', 'C', 'XC', 'L', 'XL', 'X', 'IX', 'V', 'IV', 'I']
    result = ''
    for v, s in zip(vals, syms):
        while num >= v:
            result += s
            num -= v
    return result

def _roman_to_int(s):
    vals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    for i in range(len(s)):
        if i + 1 < len(s) and vals.get(s[i], 0) < vals.get(s[i+1], 0):
            total -= vals.get(s[i], 0)
        else:
            total += vals.get(s[i], 0)
    return total

def solve_base_conversion(prompt, answer):
    examples = re.findall(r'(\d+)\s*->\s*([A-Z]+)', prompt)
    if examples:
        is_roman = all(_int_to_roman(int(n)) == r for n, r in examples)
        if is_roman:
            query = re.search(r'(?:write|convert)\s+(?:the\s+)?number\s+(\d+)', prompt)
            if query:
                num = int(query.group(1))
                predicted = _int_to_roman(num)
                remainder = num
                parts = []
                vals = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
                syms = ['M', 'CM', 'D', 'CD', 'C', 'XC', 'L', 'XL', 'X', 'IX', 'V', 'IV', 'I']
                for v, s in zip(vals, syms):
                    while remainder >= v:
                        parts.append(f"{s} ({v})")
                        remainder -= v
                cot = (
                    f"The examples show decimal to Roman numeral conversion.\n\n"
                    f"Verifying: {', '.join(f'{n} -> {r}' for n, r in examples[:3])}\n\n"
                    f"Converting {num} to Roman numerals:\n"
                    f"  {num} = {' + '.join(parts)}\n"
                    f"  Result: {predicted}\n\n\\boxed{{{predicted}}}"
                )
                return predicted, cot
    examples_rev = re.findall(r'([A-Z]+)\s*->\s*(\d+)', prompt)
    if examples_rev:
        is_roman = all(_roman_to_int(r) == int(n) for r, n in examples_rev)
        if is_roman:
            query = re.search(r'(?:write|convert)\s+(?:the\s+)?(?:number\s+)?([A-Z]+)', prompt)
            if query:
                rom = query.group(1)
                predicted = str(_roman_to_int(rom))
                cot = (
                    f"The examples show Roman numeral to decimal conversion.\n\n"
                    f"Converting {rom}:\n"
                    f"  {'  '.join(f'{c}={_roman_to_int(c)}' for c in rom)}\n"
                    f"  Total = {predicted}\n\n\\boxed{{{predicted}}}"
                )
                return predicted, cot
    return None, None

def solve_text_encryption(prompt, answer):
    is_decrypt = 'decrypt' in prompt.lower()
    examples = re.findall(r'(.+?)\s*->\s*(.+)', prompt)
    query_m = re.search(r'(?:de|en)crypt the following text:\s*(.+?)(?:\n|$)', prompt)
    if not examples or not query_m:
        return None, None
    query = query_m.group(1).strip()
    char_map = {}
    for a, b in examples:
        a, b = a.strip(), b.strip()
        if len(a) != len(b):
            continue
        for x, y in zip(a, b):
            if x == ' ' and y == ' ':
                continue
            if x in char_map and char_map[x] != y:
                return None, None
            char_map[x] = y
    if len(query) == len(answer):
        for c, p in zip(query, answer):
            if c == ' ' and p == ' ':
                continue
            if c in char_map and char_map[c] != p:
                return None, None
            char_map[c] = p
    result = ''
    for c in query:
        if c == ' ':
            result += ' '
        elif c in char_map:
            result += char_map[c]
        else:
            return None, None
    shown_mappings = {}
    for c in query:
        if c != ' ' and c in char_map:
            shown_mappings[c] = char_map[c]
    direction = "cipher -> plain" if is_decrypt else "plain -> cipher"
    table_str = ", ".join(f"'{k}'->'{v}'" for k, v in sorted(shown_mappings.items()))
    cot = (
        f"This is a substitution cipher ({direction}). I'll build the letter mapping from the examples.\n\n"
        f"From the examples, I can extract these mappings:\n  {table_str}\n\n"
        f"Applying the mapping to '{query}':\n"
    )
    words_in = query.split()
    words_out = result.split()
    for wi, wo in zip(words_in, words_out):
        mapping_detail = " ".join(f"{c}->{char_map[c]}" for c in wi)
        cot += f"  '{wi}' -> {mapping_detail} -> '{wo}'\n"
    cot += f"\n\\boxed{{{result}}}"
    return result, cot

# ═══════════════════════════════════════════════════════════════════════════════
# BIT MANIPULATION SOLVER
# ═══════════════════════════════════════════════════════════════════════════════

def _get_bit(s, pos):
    return int(s[pos])

def _solve_bit_functions(pairs):
    funcs = [None] * 8
    for out_pos in range(8):
        expected = [_get_bit(out, out_pos) for _, out in pairs]
        for in_pos in range(8):
            direct = [_get_bit(inp, in_pos) for inp, _ in pairs]
            if direct == expected:
                funcs[out_pos] = ('direct', in_pos)
                break
            if [1 - b for b in direct] == expected:
                funcs[out_pos] = ('not', in_pos)
                break
        if funcs[out_pos]:
            continue
        found = False
        for i, j in combinations(range(8), 2):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            tests = [
                ('xor', [a ^ b for a, b in zip(bi, bj)]),
                ('xnor', [1 - (a ^ b) for a, b in zip(bi, bj)]),
                ('and', [a & b for a, b in zip(bi, bj)]),
                ('nand', [1 - (a & b) for a, b in zip(bi, bj)]),
                ('or', [a | b for a, b in zip(bi, bj)]),
                ('nor', [1 - (a | b) for a, b in zip(bi, bj)]),
            ]
            for name, result in tests:
                if result == expected:
                    funcs[out_pos] = (name, i, j)
                    found = True
                    break
            if found:
                break
        if funcs[out_pos]:
            continue
        for i, j, k in combinations(range(8), 3):
            bi = [_get_bit(inp, i) for inp, _ in pairs]
            bj = [_get_bit(inp, j) for inp, _ in pairs]
            bk = [_get_bit(inp, k) for inp, _ in pairs]
            tests = [
                ('majority', [1 if (a+b+c) >= 2 else 0 for a, b, c in zip(bi, bj, bk)]),
                ('minority', [1 if (a+b+c) < 2 else 0 for a, b, c in zip(bi, bj, bk)]),
                ('choice', [b if a == 1 else c for a, b, c in zip(bi, bj, bk)]),
                ('choice_inv', [b if a == 0 else c for a, b, c in zip(bi, bj, bk)]),
            ]
            for name, result in tests:
                if result == expected:
                    funcs[out_pos] = (name, i, j, k)
                    found = True
                    break
            if found:
                break
    return funcs

def _apply_bit_func(func, query):
    if func is None:
        return None
    name = func[0]
    if name == 'direct':
        return _get_bit(query, func[1])
    elif name == 'not':
        return 1 - _get_bit(query, func[1])
    elif name in ('xor', 'xnor', 'and', 'nand', 'or', 'nor'):
        a, b = _get_bit(query, func[1]), _get_bit(query, func[2])
        if name == 'xor': return a ^ b
        elif name == 'xnor': return 1 - (a ^ b)
        elif name == 'and': return a & b
        elif name == 'nand': return 1 - (a & b)
        elif name == 'or': return a | b
        elif name == 'nor': return 1 - (a | b)
    elif name in ('majority', 'minority', 'choice', 'choice_inv'):
        a, b, c = _get_bit(query, func[1]), _get_bit(query, func[2]), _get_bit(query, func[3])
        if name == 'majority': return 1 if (a+b+c) >= 2 else 0
        elif name == 'minority': return 1 if (a+b+c) < 2 else 0
        elif name == 'choice': return b if a == 1 else c
        elif name == 'choice_inv': return b if a == 0 else c
    return None

def _describe_bit_func(func):
    if func is None:
        return "unknown"
    name = func[0]
    if name == 'direct':
        return f"input[{func[1]}]"
    elif name == 'not':
        return f"NOT input[{func[1]}]"
    elif name in ('xor', 'xnor', 'and', 'nand', 'or', 'nor'):
        return f"input[{func[1]}] {name.upper()} input[{func[2]}]"
    elif name in ('majority', 'minority'):
        return f"{name}(input[{func[1]}], input[{func[2]}], input[{func[3]}])"
    elif name == 'choice':
        return f"if input[{func[1]}] then input[{func[2]}] else input[{func[3]}]"
    elif name == 'choice_inv':
        return f"if NOT input[{func[1]}] then input[{func[2]}] else input[{func[3]}]"
    return str(func)

def solve_bit_manipulation(prompt, answer):
    pairs = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)
    query_m = re.search(r'output for:\s*([01]{8})', prompt)
    if not pairs or not query_m:
        return None, None
    query = query_m.group(1)
    funcs = _solve_bit_functions(pairs)
    predicted_bits = [_apply_bit_func(f, query) for f in funcs]
    all_solved = all(b is not None for b in predicted_bits)
    if all_solved:
        predicted = ''.join(str(b) for b in predicted_bits)
        if predicted == answer:
            cot = "I need to find the secret bit transformation by analyzing each output bit position.\n\n"
            cot += f"Given {len(pairs)} input->output examples, I'll determine what function produces each output bit.\n\n"
            simple_bits = [(i, f) for i, f in enumerate(funcs) if f[0] == 'direct']
            not_bits = [(i, f) for i, f in enumerate(funcs) if f[0] == 'not']
            complex_bits = [(i, f) for i, f in enumerate(funcs) if f[0] not in ('direct', 'not')]
            if simple_bits:
                cot += "Direct bit mappings (output = input bit):\n"
                for pos, f in simple_bits:
                    cot += f"  output[{pos}] = input[{f[1]}]\n"
                cot += "\n"
            if not_bits:
                cot += "Inverted bit mappings (output = NOT input bit):\n"
                for pos, f in not_bits:
                    cot += f"  output[{pos}] = NOT input[{f[1]}]\n"
                cot += "\n"
            if complex_bits:
                cot += "Complex bit operations:\n"
                for pos, f in complex_bits:
                    cot += f"  output[{pos}] = {_describe_bit_func(f)}\n"
                cot += "\n"
            if len(pairs) > 0:
                inp0, out0 = pairs[0]
                cot += f"Verification with first example: {inp0} -> {out0}\n\n"
            cot += f"Applying to query {query}:\n"
            for i in range(8):
                cot += f"  bit[{i}]: {_describe_bit_func(funcs[i])} = {predicted_bits[i]}\n"
            cot += f"\nResult: {predicted}\n\n\\boxed{{{predicted}}}"
            return predicted, cot
    # Scaffold for unsolved cases
    cot = "I need to find the secret bit transformation by analyzing each output bit position.\n\n"
    cot += f"Given {len(pairs)} input->output pairs:\n"
    for inp, out in pairs[:4]:
        cot += f"  {inp} -> {out}\n"
    if len(pairs) > 4:
        cot += f"  ... and {len(pairs) - 4} more examples\n"
    cot += "\nAnalyzing each output bit as a function of input bits:\n"
    for i in range(8):
        if funcs[i] is not None:
            cot += f"  output[{i}] = {_describe_bit_func(funcs[i])}\n"
        else:
            cot += f"  output[{i}] = complex function (multi-bit dependency)\n"
    cot += f"\nApplying the full transformation to query {query}:\n"
    cot += f"Result: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ═══════════════════════════════════════════════════════════════════════════════
# EQUATION TRANSFORMATION SOLVER
# ═══════════════════════════════════════════════════════════════════════════════

_EQ_OPERATIONS = {
    'addition': lambda a, b: str(a + b),
    'subtraction (A-B)': lambda a, b: str(a - b),
    'subtraction (B-A)': lambda a, b: str(b - a),
    'absolute difference': lambda a, b: str(abs(a - b)),
    'multiplication': lambda a, b: str(a * b),
    'concatenation (AB)': lambda a, b: str(a) + str(b),
    'concatenation (BA)': lambda a, b: str(b) + str(a),
    'integer division (A/B)': lambda a, b: str(a // b) if b != 0 else None,
    'integer division (B/A)': lambda a, b: str(b // a) if a != 0 else None,
    'modulo (A%B)': lambda a, b: str(a % b) if b != 0 else None,
    'modulo (B%A)': lambda a, b: str(b % a) if a != 0 else None,
    'XOR': lambda a, b: str(a ^ b),
    'bitwise AND': lambda a, b: str(a & b),
    'bitwise OR': lambda a, b: str(a | b),
    'max': lambda a, b: str(max(a, b)),
    'min': lambda a, b: str(min(a, b)),
}

def _parse_eq_examples(prompt):
    lines = prompt.strip().split('\n')
    examples = []
    query = None
    for line in lines:
        line = line.strip()
        if 'determine the result for:' in line.lower():
            m = re.search(r'determine the result for:\s*(.+)', line, re.IGNORECASE)
            if m:
                query = m.group(1).strip()
        elif ' = ' in line and 'wonderland' not in line.lower() and 'transformation' not in line.lower() and 'examples' not in line.lower():
            parts = line.split(' = ', 1)
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))
    return examples, query

def solve_equation_transformation(prompt, answer):
    examples, query = _parse_eq_examples(prompt)
    if not examples or not query or len(query) != 5:
        return None, None
    parsed = []
    all_valid = True
    for lhs, rhs in examples:
        if len(lhs) != 5:
            all_valid = False
            break
        parsed.append((lhs[:2], lhs[2], lhs[3:], rhs))
    if not all_valid or not parsed:
        return None, None
    q_a, q_op, q_b = query[:2], query[2], query[3:]
    is_numeric = all(a.isdigit() and b.isdigit() for a, _, b, _ in parsed) and q_a.isdigit() and q_b.isdigit()
    if is_numeric:
        by_op = {}
        for a, op, b, rhs in parsed:
            by_op.setdefault(op, []).append((int(a), int(b), rhs))
        op_mapping = {}
        for op_char, op_examples in by_op.items():
            for op_name, op_func in _EQ_OPERATIONS.items():
                if all(op_func(a, b) == rhs for a, b, rhs in op_examples):
                    op_mapping[op_char] = op_name
                    break
        if q_op in op_mapping:
            op_name = op_mapping[q_op]
            predicted = _EQ_OPERATIONS[op_name](int(q_a), int(q_b))
            if predicted == answer:
                cot = "I need to figure out what each operator symbol means by testing the examples.\n\n"
                cot += "Parsing the examples (format: A operator B = result):\n"
                for a, op, b, rhs in parsed:
                    cot += f"  {a} '{op}' {b} = {rhs}\n"
                cot += "\nTesting each operator against standard operations:\n"
                for op_char, op_name in op_mapping.items():
                    op_examples = by_op[op_char]
                    cot += f"  Operator '{op_char}' = {op_name}:\n"
                    for a, b, rhs in op_examples[:2]:
                        cot += f"    {a} {op_name.split('(')[0].strip()} {b} = {rhs}\n"
                cot += f"\nApplying to query: {q_a} '{q_op}' ({op_mapping[q_op]}) {q_b}\n"
                cot += f"Result = {predicted}\n\n\\boxed{{{predicted}}}"
                return predicted, cot
    # Scaffold fallback
    cot = "I need to figure out the secret transformation rules from the examples.\n\n"
    cot += "Each expression has the format: operand1 (2 chars) + operator (1 char) + operand2 (2 chars) = result.\n\n"
    cot += "Given examples:\n"
    for a, op, b, rhs in parsed:
        cot += f"  {a} '{op}' {b} = {rhs}\n"
    by_op = {}
    for a, op, b, rhs in parsed:
        by_op.setdefault(op, []).append((a, b, rhs))
    cot += f"\nOperators used: {list(by_op.keys())}\n"
    cot += f"\nApplying the identified rule to the query: {q_a} '{q_op}' {q_b}\n"
    cot += f"Result: {answer}\n\n\\boxed{{{answer}}}"
    return answer, cot

# ═══════════════════════════════════════════════════════════════════════════════
# FALLBACK TEMPLATES
# ═══════════════════════════════════════════════════════════════════════════════

FALLBACK_COT_TEMPLATES = {
    'Equation Transformation': (
        "I need to figure out the secret transformation rules from the examples.\n\n"
        "Each expression has format: operand1 operator operand2 = result.\n"
        "I need to determine what each operator does by analyzing the input-output pairs.\n\n"
        "After analyzing all the examples and identifying the pattern:\n\n"
        "\\boxed{{{answer}}}"
    ),
    'Bit Manipulation': (
        "I need to find the secret bit transformation by analyzing each output bit position.\n\n"
        "Looking at the 8-bit binary input->output pairs, I need to determine what operation "
        "produces each output bit. Common operations include bit shifts, rotations, XOR, AND, OR, NOT, "
        "and combinations like majority or choice functions.\n\n"
        "Testing each output bit against input bits to find the mapping:\n"
        "After systematic analysis of all 8 bit positions:\n\n"
        "\\boxed{{{answer}}}"
    ),
}

# ═══════════════════════════════════════════════════════════════════════════════
# PUZZLE TYPE DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

def detect_puzzle_type(prompt):
    p = prompt.lower()
    if 'gravitational' in p or 'd = 0.5*g*t^2' in p or 'd = 0.5*g*t' in p:
        return 'Gravitational Constant'
    elif 'unit' in p and 'conver' in p and 'becomes' in p:
        return 'Unit Conversion'
    elif ('roman' in p or ('number' in p and 'convert' in p and '->' in prompt)) and re.search(r'\d+\s*->\s*[A-Z]+|[A-Z]+\s*->\s*\d+', prompt):
        return 'Number Base Conversion'
    elif ('encrypt' in p or 'decrypt' in p) and '->' in prompt:
        return 'Text Encryption'
    elif 'bit manipulation' in p or (re.search(r'[01]{8}\s*->', prompt) and 'output for:' in p):
        return 'Bit Manipulation'
    elif 'transformation' in p and ('=' in prompt) and re.search(r'determine the result for:', p):
        return 'Equation Transformation'
    return 'Unknown'

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN: Build SFT training data with solver-backed CoT
# ═══════════════════════════════════════════════════════════════════════════════

SOLVER_MAP = {
    'Gravitational Constant': solve_gravity,
    'Unit Conversion': solve_unit_conversion,
    'Number Base Conversion': solve_base_conversion,
    'Text Encryption': solve_text_encryption,
    'Bit Manipulation': solve_bit_manipulation,
    'Equation Transformation': solve_equation_transformation,
}

solver_stats = {'solved': 0, 'fallback': 0, 'total': 0}
type_counts = {}

def build_sft_messages(row):
    """Build chat messages for SFT with solver-backed CoT."""
    prompt = row["prompt"]
    answer = str(row["answer"])
    puzzle_type = detect_puzzle_type(prompt)
    type_counts[puzzle_type] = type_counts.get(puzzle_type, 0) + 1
    solver_stats['total'] += 1

    assistant_msg = None
    if puzzle_type in SOLVER_MAP:
        predicted, cot = SOLVER_MAP[puzzle_type](prompt, answer)
        if cot is not None:
            assistant_msg = cot
            solver_stats['solved'] += 1

    if assistant_msg is None:
        solver_stats['fallback'] += 1
        template = FALLBACK_COT_TEMPLATES.get(
            puzzle_type,
            "After analyzing the pattern from the examples:\n\n\\boxed{{{answer}}}"
        )
        assistant_msg = template.format(answer=answer)

    user_content = prompt + BOXED_INSTRUCTION
    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_msg},
        ]
    }

# Build SFT dataset
train_records = [build_sft_messages(row) for _, row in train_df.iterrows()]
train_dataset = Dataset.from_list(train_records)

print(f"SFT dataset: {len(train_dataset):,} examples")
print(f"  Solver-backed CoT: {solver_stats['solved']}")
print(f"  Fallback template: {solver_stats['fallback']}")
print(f"\nPuzzle type distribution:")
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")
print(f"\nSample:\n{train_dataset[0]['messages']}")

# ═══════════════════════════════════════════════════════════════════════════════
# GRPO DATASET — prompt only, no gold answer in text
# ═══════════════════════════════════════════════════════════════════════════════

grpo_subsample = train_df.sample(n=min(GRPO_SUBSAMPLE_SIZE, len(train_df)), random_state=42)
grpo_records = []
for _, row in grpo_subsample.iterrows():
    grpo_records.append({
        "prompt": [
            {"role": "user", "content": row["prompt"] + BOXED_INSTRUCTION},
        ],
        "ground_truth": str(row["answer"]),
    })
grpo_dataset = Dataset.from_list(grpo_records)
print(f"\nGRPO dataset: {len(grpo_dataset)} prompts (each generates {GRPO_NUM_GENERATIONS} completions)")

## 4. Load Base Model with LoRA (4-bit via Unsloth)

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = LOAD_IN_4BIT,  # False for this MoE model (see LOAD_IN_4BIT comment above)
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    unsloth_force_compile = False,
    attn_implementation = "eager",
    torch_dtype=torch.bfloat16,
    dtype=None,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.10: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.


In [7]:
# del model, tokenizer

# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [8]:
target_modules = [
    'out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens',
    'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj'
]

In [9]:
if FRESH_START:
    # Apply LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",  # saves VRAM
        random_state=42,
    )

model.print_trainable_parameters()

Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens', 'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 888,154,112 || all params: 32,466,091,456 || trainable%: 2.7356


## 5. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = train_dataset.map(lambda ex: {
    "text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,
    args=TrainingArguments(
        output_dir="./nemotron-lora-checkpoints",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=True,
        logging_steps=50,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
        report_to="none",
        dataloader_num_workers=4,
        eval_strategy="no",
    ),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    dataset_kwargs={"skip_prepare_dataset": False},
)

print("Starting SFT training...")
trainer.train()
print("SFT training complete.")

## 7. Save Final Adapter (SFT + GRPO)

In [11]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Verify adapter_config.json is present (required by competition)
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    "adapter_config.json missing!"

print(f"Adapter saved to ./{ADAPTER_DIR}/")
print("Files:", os.listdir(ADAPTER_DIR))

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Adapter saved to ./nemotron-lora-adapter/
Files: ['adapter_model.safetensors', 'README.md', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'adapter_config.json']


## 6b. GRPO Training — RL Refinement Phase

After SFT warmup, we refine the LoRA with **Group Relative Policy Optimization (GRPO)**.

The model generates multiple completions per prompt. Three reward functions score each:
1. **Cosine reward** — accuracy scaled by length (short correct = high reward)
2. **Format reward** — did it produce `\boxed{}`?
3. **Length reward** — penalizes verbose reasoning

`beta=0` means no reference model is needed (saves ~50% memory).

In [ ]:
import gc
from trl import GRPOTrainer, GRPOConfig

# Free SFT trainer memory
del trainer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU memory after SFT cleanup: {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated")

# ═══════════════════════════════════════════════════════════════════════════════
# GRPO REWARD FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

_reward_debug_counter = {"calls": 0}

def _normalize_answer(s):
    s = s.strip()
    try:
        f = float(s)
        if f == int(f):
            return str(int(f))
        return str(f)
    except (ValueError, OverflowError):
        return s

def _extract_boxed(content):
    match = re.search(r'\\boxed\{([^}]*)\}', content, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r'boxed\{([^}]*)\}', content, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def _get_content(completion):
    if isinstance(completion, list):
        return completion[-1]["content"] if completion else ""
    return completion

def cosine_reward(completions, ground_truth, **kwargs):
    """Cosine-scaled accuracy reward (from Light-R1).
    Correct + short -> ~1.0, Correct + long -> ~0.1
    Wrong -> negative scaled by length.
    """
    max_len = GRPO_MAX_COMPLETION
    rewards = []
    _reward_debug_counter["calls"] += 1
    show_debug = _reward_debug_counter["calls"] <= 2

    for i, (completion, gt) in enumerate(zip(completions, ground_truth)):
        content = _get_content(completion)
        extracted = _extract_boxed(content)
        clen = len(content)
        progress = min(clen / max(max_len, 1), 1.0)
        cos_scale = 0.5 * (1.0 + math.cos(math.pi * progress))

        if extracted is not None and _normalize_answer(extracted) == _normalize_answer(gt):
            reward = 0.1 + 0.9 * cos_scale
        elif extracted is not None:
            reward = -0.1 - 0.9 * (1.0 - cos_scale)
        else:
            reward = -0.5 * progress

        rewards.append(reward)

        if show_debug and i < 2:
            tail = content[-120:] if len(content) > 120 else content
            print(f"  [COSINE] extracted={extracted!r}, gt={gt.strip()!r}, "
                  f"reward={reward:.3f}, len={clen}", flush=True)

    return rewards

def format_reward(completions, **kwargs):
    """Binary format reward: 1.0 if \\boxed{} present, 0.0 otherwise."""
    rewards = []
    for completion in completions:
        content = _get_content(completion)
        rewards.append(1.0 if _extract_boxed(content) is not None else 0.0)
    return rewards

def length_reward(completions, **kwargs):
    """Length penalty: 0.0 for empty, scales to -1.0 at max_completion_length."""
    max_len = GRPO_MAX_COMPLETION
    rewards = []
    for completion in completions:
        content = _get_content(completion)
        progress = min(len(content) / max(max_len, 1), 1.0)
        rewards.append(-progress)
    return rewards

# Sanity check
_reward_debug_counter["calls"] = 0
_test_cos = cosine_reward(
    completions=["The answer is \\boxed{42}.", "No boxed answer here.", "Wrong: \\boxed{99}." + " " * 800],
    ground_truth=["42", "42", "42"],
)
_test_fmt = format_reward(
    completions=["The answer is \\boxed{42}.", "No boxed answer here.", "Wrong: \\boxed{99}."],
)
_reward_debug_counter["calls"] = 0
print(f"Cosine reward sanity: {[f'{r:.3f}' for r in _test_cos]}  (expect ~[1.0, 0.0, ~-0.7])")
print(f"Format reward sanity: {_test_fmt}  (expect [1.0, 0.0, 1.0])")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRPO TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

# Switch model back to training mode
FastLanguageModel.for_training(model)

# GRPO requires left-padding for generation
tokenizer.padding_side = "left"

grpo_config = GRPOConfig(
    output_dir="./nemotron-grpo-checkpoints",
    num_generations=GRPO_NUM_GENERATIONS,
    max_completion_length=GRPO_MAX_COMPLETION,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=GRPO_EPOCHS,
    learning_rate=GRPO_LR,
    temperature=GRPO_TEMPERATURE,
    beta=0.0,                    # no KL penalty -> no reference model needed
    loss_type="grpo",
    logging_steps=2,
    max_grad_norm=0.1,           # aggressive clipping for stability
    weight_decay=0.1,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    remove_unused_columns=False, # keep ground_truth column for reward funcs
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[cosine_reward, format_reward, length_reward],
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    args=grpo_config,
)

print(f"GRPO config: {GRPO_NUM_GENERATIONS} generations/prompt, temp={GRPO_TEMPERATURE}, lr={GRPO_LR}")
print(f"GRPO dataset: {len(grpo_dataset)} prompts")
print("Starting GRPO training...")
grpo_trainer.train()
print("GRPO training complete.")

## 8. Local Validation

Run inference on a small held-out slice of the training set to get a quick local accuracy estimate before submitting.

In [12]:
def extract_final_answer(text: str | None) -> str:
    r"""Extract the final answer from model output, prioritising \boxed{}."""
    if text is None:
        return "NOT_FOUND"

    # Prefer \boxed{...}
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()

    # Common fallback patterns
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.findall(pattern, text, re.IGNORECASE)
        if m:
            return m[-1].strip()

    # Last numeric value
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    # Last non-empty line
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Return True if predicted matches stored_answer (numeric or string)."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    try:
        return math.isclose(float(stored_answer), float(predicted),
                            rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


print("Helper functions ready.")

Helper functions ready.


In [13]:
# # Switch model to inference mode (Unsloth optimisation)
# FastLanguageModel.for_inference(model)

# # Sample 100 examples from train for local eval
# eval_df = train_df.sample(20, random_state=42).reset_index(drop=True)

# correct = 0
# for _, row in eval_df.iterrows():
#     user_content = row["prompt"] + BOXED_INSTRUCTION
#     # Apply the tokenizer's chat template (system prompt optional for baseline)
#     messages = [{"role": "user", "content": user_content}]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

#     output_ids = model.generate(
#         input_ids=input_ids,
#         max_new_tokens=MAX_NEW_TOKENS,
#         temperature=TEMPERATURE if TEMPERATURE > 0 else None,
#         do_sample=TEMPERATURE > 0,
#         pad_token_id=tokenizer.eos_token_id,
#     )
#     # Decode only newly generated tokens
#     generated = tokenizer.decode(
#         output_ids[0][input_ids.shape[1]:],
#         skip_special_tokens=True
#     )

#     pred = extract_final_answer(generated)
#     if verify(str(row["answer"]), pred):
#         correct += 1

# local_acc = correct / len(eval_df)
# print(f"Local accuracy on {len(eval_df)} train samples: {local_acc:.2%}")

## 9. Submission

The competition expects a zip archive containing the LoRA adapter directory (with `adapter_config.json` at the root of the archive or a sub-directory).

In [ ]:
!rm -rf /kaggle/working/nemotron-lora-checkpoints
!rm -rf /kaggle/working/nemotron-grpo-checkpoints

In [15]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(ADAPTER_DIR):
        zf.write(
            os.path.join(ADAPTER_DIR, fname),
            arcname=os.path.join(ADAPTER_DIR, fname),
        )

# Verify
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = zf.namelist()

has_config = any("adapter_config.json" in n for n in names)
print(f"Files in {SUBMISSION_ZIP}: {names}")
print(f"adapter_config.json present: {has_config}")

Files in submission.zip: ['nemotron-lora-adapter/adapter_model.safetensors', 'nemotron-lora-adapter/README.md', 'nemotron-lora-adapter/chat_template.jinja', 'nemotron-lora-adapter/tokenizer_config.json', 'nemotron-lora-adapter/tokenizer.json', 'nemotron-lora-adapter/adapter_config.json']
adapter_config.json present: True
